In [1]:
library(lme4)
library(lmerTest)
library(car)
library(ggplot2)

Loading required package: Matrix

Warning message:
“package ‘Matrix’ was built under R version 4.4.3”
Warning message:
“package ‘lmerTest’ was built under R version 4.4.3”

Attaching package: ‘lmerTest’


The following object is masked from ‘package:lme4’:

    lmer


The following object is masked from ‘package:stats’:

    step


Loading required package: carData



In [2]:
df <- read.csv("concatenated_data/experiment2_results.csv")
df <- df[df$preceding_tone %in% c("tone51", "tone55"),]
df$lang <- as.factor(df$lang)
df$participant <- as.factor(df$participant)
df$word_freq_ratio <- scale(df$word_freq_ratio)

df$preceding_tone_offset <- as.integer(sub(".*(\\d)$", "\\1", df$preceding_tone))

df$level <- as.numeric(scale(df$level, center = TRUE, scale = FALSE))
df$preceding_tone_offset <- as.numeric(scale(df$preceding_tone_offset, center = TRUE, scale = FALSE))
df$fluency <- as.numeric(scale(df$fluency, center = TRUE, scale = FALSE))

cor(df$word_freq_ratio, df$falling_tone_chosen, method = "pearson")
cor(df$word_freq_ratio, df$preceding_tone_offset, method = "pearson")

0.0104847


0.1308531


In [3]:
temp <- df
model <- glmer(falling_tone_chosen ~ level*lang + preceding_tone_offset*lang + (1|participant), family=binomial, data=temp, control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e5)))
summary(model)

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: falling_tone_chosen ~ level * lang + preceding_tone_offset *  
    lang + (1 | participant)
   Data: temp
Control: glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e+05))

     AIC      BIC   logLik deviance df.resid 
  2527.3   2596.0  -1253.7   2507.3     7090 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-329.47   -0.09    0.00    0.10  147.50 

Random effects:
 Groups      Name        Variance Std.Dev.
 participant (Intercept) 0.278    0.5273  
Number of obs: 7100, groups:  participant, 43

Fixed effects:
                                        Estimate Std. Error z value Pr(>|z|)
(Intercept)                              -1.0805     0.2108  -5.125 2.97e-07
level                                     2.8932     0.2180  13.272  < 2e-16
langman_bilingual                         0.7248     0.2484   2.918 0.003524
langmin           

In [4]:
temp$residual <- residuals(model)
model <- lmer(residual ~ word_freq_ratio + (1|participant), data=temp)
summary(model) # Post-hoc residual analysis to see if word frequency ratio is a significant predictor of the remaining variance

boundary (singular) fit: see help('isSingular')



Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: residual ~ word_freq_ratio + (1 | participant)
   Data: temp

REML criterion at convergence: 12547.5

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-8.2261 -0.2215  0.0046  0.2375  7.6482 

Random effects:
 Groups      Name        Variance Std.Dev.
 participant (Intercept) 0.0000   0.0000  
 Residual                0.3421   0.5849  
Number of obs: 7100, groups:  participant, 43

Fixed effects:
                  Estimate Std. Error         df t value Pr(>|t|)
(Intercept)     -1.889e-03  6.942e-03  7.098e+03  -0.272    0.786
word_freq_ratio  2.050e-03  6.942e-03  7.098e+03   0.295    0.768

Correlation of Fixed Effects:
            (Intr)
word_frq_rt 0.000 
optimizer (nloptwrap) convergence code: 0 (OK)
boundary (singular) fit: see help('isSingular')


In [5]:
temp <- df
temp <- temp[(temp$lang =="min") | (temp$lang =="man_bilingual"),]
temp$lang <- droplevels(temp$lang)
print(contrasts(temp$lang))
model <- glmer(falling_tone_chosen ~ level*lang + preceding_tone_offset*lang + (1|participant), family=binomial, data=temp, control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e5)))
summary(model)

              min
man_bilingual   0
min             1


Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: falling_tone_chosen ~ level * lang + preceding_tone_offset *  
    lang + (1 | participant)
   Data: temp
Control: glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e+05))

     AIC      BIC   logLik deviance df.resid 
  2170.0   2216.4  -1078.0   2156.0     5593 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-329.72   -0.12    0.00    0.12   43.91 

Random effects:
 Groups      Name        Variance Std.Dev.
 participant (Intercept) 0.2791   0.5283  
Number of obs: 5600, groups:  participant, 28

Fixed effects:
                              Estimate Std. Error z value Pr(>|z|)    
(Intercept)                   -0.35569    0.13171  -2.701  0.00692 ** 
level                          1.99048    0.09122  21.822  < 2e-16 ***
langmin                        0.70811    0.11450   6.184 6.24e-10 ***
preceding_tone_offset         -1.11634    

In [6]:
unique(df$lang)

[1] man           man_bilingual min          
Levels: man man_bilingual min

In [7]:
# Discussion on bilingualism: Does fluency in TSM affect perceptual compensation in TM?
temp <- df
temp <- temp[(temp$lang =="man_bilingual"),]
temp$lang <- droplevels(temp$lang)
model <- glmer(falling_tone_chosen ~ level*fluency + preceding_tone_offset*fluency + (1|participant), family=binomial, data=temp, control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e5)))
summary(model)

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: falling_tone_chosen ~ level * fluency + preceding_tone_offset *  
    fluency + (1 | participant)
   Data: temp
Control: glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e+05))

     AIC      BIC   logLik deviance df.resid 
   923.8    965.4   -454.9    909.8     2793 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-597.63   -0.07    0.00    0.07   58.73 

Random effects:
 Groups      Name        Variance Std.Dev.
 participant (Intercept) 0.5887   0.7673  
Number of obs: 2800, groups:  participant, 28

Fixed effects:
                              Estimate Std. Error z value Pr(>|z|)    
(Intercept)                   -0.19758    0.22218  -0.889   0.3738    
level                          2.29458    0.14929  15.370   <2e-16 ***
fluency                       -0.11196    0.09709  -1.153   0.2489    
preceding_tone_offset         -1.226

In [8]:
temp <- df
temp <- temp[(temp$lang =="min"),]
temp$lang <- droplevels(temp$lang)
model <- glmer(falling_tone_chosen ~ level + fluency + preceding_tone_offset*fluency + (1|participant), family=binomial, data=temp, control = glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e5)))
summary(model)

Generalized linear mixed model fit by maximum likelihood (Laplace
  Approximation) [glmerMod]
 Family: binomial  ( logit )
Formula: falling_tone_chosen ~ level + fluency + preceding_tone_offset *  
    fluency + (1 | participant)
   Data: temp
Control: glmerControl(optimizer = "bobyqa", optCtrl = list(maxfun = 1e+05))

     AIC      BIC   logLik deviance df.resid 
  1231.5   1267.1   -609.8   1219.5     2794 

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-81.013  -0.151   0.015   0.152  41.747 

Random effects:
 Groups      Name        Variance Std.Dev.
 participant (Intercept) 0.2732   0.5226  
Number of obs: 2800, groups:  participant, 28

Fixed effects:
                              Estimate Std. Error z value Pr(>|z|)    
(Intercept)                    0.28177    0.16035   1.757   0.0789 .  
level                          1.53985    0.06473  23.787   <2e-16 ***
fluency                        0.04912    0.07048   0.697   0.4858    
preceding_tone_offset         -0.563